In [ ]:
!pip install -q dotenv langdetect langchain-community rouge_score bert_score sentence-transformers transformers accelerate bitsandbytes peft datasets evaluate neo4j langchain arabic-reshaper python-bidi textwrap3


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.0/444.0 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is t

In [ ]:
%%writefile GRAPHRAG_MISTRAL.py
import os
import re
import torch
import json
import numpy as np
from typing import Dict, List, Optional, Tuple, Any
from neo4j import GraphDatabase
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig,
    AutoModel,
    AutoTokenizer as SentenceTokenizer
)
from sentence_transformers import SentenceTransformer, util
from peft import PeftModel, PeftConfig
import arabic_reshaper
from bidi.algorithm import get_display
from unicodedata import normalize
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
import logging
import gc
import asyncio
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import threading
import time
from functools import lru_cache
import hashlib

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Constants
NEO4J_URI = os.getenv("NEO4J_URI", "")
NEO4J_USER = os.getenv("NEO4J_USER", "")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", " ")
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
LORA_MODEL_PATH = ""
SEMANTIC_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"


GLOBAL_MODELS = {}
GLOBAL_RETRIEVER = None
GLOBAL_MODELS_LOCK = threading.Lock()

class UrduTextFormatter:
    @staticmethod
    def format(text: str, width: int = 80) -> str:
        """Format Urdu text with proper shaping and bidirectional support"""
        if not text or not any('\u0600' <= c <= '\u06FF' for c in text):
            return text

        try:
            text = normalize('NFC', text)
            text = UrduTextFormatter._fix_urdu_encoding(text)

            config = arabic_reshaper.config_for_language('urdu')
            config['use_unshaped_instead_of_isolated'] = False
            reshaper = arabic_reshaper.ArabicReshaper(config)

            reshaped = reshaper.reshape(text)
            bidi_text = get_display(reshaped)

            words = bidi_text.split()
            reversed_words = list(reversed(words))

            lines = []
            current_line = []
            current_length = 0

            for word in reversed_words:
                word_length = len(word)
                if current_length + word_length <= width or not current_line:
                    current_line.append(word)
                    current_length += word_length + 1
                else:
                    lines.append(' '.join(current_line))
                    current_line = [word]
                    current_length = word_length + 1

            if current_line:
                lines.append(' '.join(current_line))

            return '\n'.join(lines)

        except Exception as e:
            logger.error(f"UrduTextFormatter Error: {e}")
            return text

    @staticmethod
    def _fix_urdu_encoding(text: str) -> str:
        """Fix common Urdu encoding issues"""
        replacements = {
            '\u0649': '\u06cc',  
            '\u064a': '\u06cc', 
            '\u0621': '\u0626',  
            '\u0643': '\u06a9',  
        }

        for old, new in replacements.items():
            text = text.replace(old, new)
        return text

    @staticmethod
    def preprocess_urdu_text(text: str) -> str:
        """Preprocess Urdu text for better model understanding"""
        if not text:
            return text

        text = normalize('NFC', text)
        text = UrduTextFormatter._fix_urdu_encoding(text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

class Language:
    """Language detection and handling"""
    ENGLISH = "english"
    URDU = "urdu"

    @classmethod
    def detect(cls, text: str) -> str:
        """Improved language detection with threshold"""
        urdu_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
        return cls.URDU if urdu_chars > max(2, len(text)*0.3) else cls.ENGLISH

class SemanticRetriever:
    """Optimized BERT-based semantic retriever with caching"""

    def __init__(self, model_name: str = SEMANTIC_MODEL_NAME):
        with GLOBAL_MODELS_LOCK:
            if 'semantic_model' in GLOBAL_MODELS:
                self.model = GLOBAL_MODELS['semantic_model']
                logger.info("Using cached semantic model")
            else:
                
                self.model = SentenceTransformer(
                    model_name,
                    device='cuda' if torch.cuda.is_available() else 'cpu'
                )
                GLOBAL_MODELS['semantic_model'] = self.model
                logger.info("Loaded new semantic model")

        self.cache = {}
        self.cache_lock = threading.Lock()
        self.batch_size = 32  

    def _generate_cache_key(self, text: str) -> str:
        """Generate consistent cache key"""
        return hashlib.md5(text.encode()).hexdigest()

    def get_embedding_batch(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for multiple texts efficiently"""
        if not texts:
            return np.array([])

        # Check cache first
        cache_keys = [self._generate_cache_key(text) for text in texts]
        cached_embeddings = []
        uncached_texts = []
        uncached_indices = []

        with self.cache_lock:
            for i, (text, cache_key) in enumerate(zip(texts, cache_keys)):
                if cache_key in self.cache:
                    cached_embeddings.append(self.cache[cache_key])
                else:
                    uncached_texts.append(text)
                    uncached_indices.append(i)

        # Process uncached texts in batches
        if uncached_texts:
            # Preprocess texts
            clean_texts = [
                re.sub(r'[^\w\s\u0600-\u06FF]', '', text.lower().strip())
                for text in uncached_texts
            ]

            
            uncached_embeddings = self.model.encode(
                clean_texts,
                batch_size=self.batch_size,
                convert_to_tensor=False,
                show_progress_bar=False
            )

            # Update cache
            with self.cache_lock:
                for i, (text, embedding) in enumerate(zip(uncached_texts, uncached_embeddings)):
                    cache_key = self._generate_cache_key(text)
                    self.cache[cache_key] = embedding
                    cached_embeddings.insert(uncached_indices[i], embedding)

        return np.array(cached_embeddings)

    def get_embedding(self, text: str) -> np.ndarray:
        """Single text embedding wrapper"""
        return self.get_embedding_batch([text])[0]

    def semantic_similarity_batch(self, queries: List[str], candidates: List[str]) -> np.ndarray:
        """Calculate semantic similarity in batch mode"""
        if not queries or not candidates:
            return np.array([])

        
        all_texts = list(set(queries + candidates))
        embeddings_dict = {}

        embeddings = self.get_embedding_batch(all_texts)
        for text, embedding in zip(all_texts, embeddings):
            embeddings_dict[text] = embedding

        # Calculate similarities
        similarities = np.zeros((len(queries), len(candidates)))
        for i, query in enumerate(queries):
            query_emb = embeddings_dict[query]
            for j, candidate in enumerate(candidates):
                cand_emb = embeddings_dict[candidate]
                if np.all(query_emb == 0) or np.all(cand_emb == 0):
                    similarities[i, j] = 0.0
                else:
                    similarities[i, j] = cosine_similarity([query_emb], [cand_emb])[0][0]

        return similarities

    def semantic_similarity(self, text1: str, text2: str) -> float:
        """Single similarity wrapper"""
        similarities = self.semantic_similarity_batch([text1], [text2])
        return similarities[0, 0] if similarities.size > 0 else 0.0

class GraphRAGRetriever:
    """Optimized Neo4j GraphRAG retriever with efficient queries"""

    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.cache = {}
        self.cache_lock = threading.Lock()
        self.semantic_retriever = SemanticRetriever()
        self.entity_cache = []
        self._precache_entities()

    def _precache_entities(self):
        """Pre-cache entity names for faster semantic search"""
        logger.info("Pre-caching entity names...")
        try:
            with self.driver.session() as session:
               
                query = """
                MATCH (n)
                WHERE n.name IS NOT NULL OR n.description IS NOT NULL
                RETURN
                    labels(n)[0] as label,
                    COALESCE(n.name, n.description) as name,
                    n.language as lang
                UNION
                MATCH (n:UnaniFormulation)-[r]->(m)
                WHERE m.name IS NOT NULL OR m.description IS NOT NULL
                RETURN
                    labels(m)[0] as label,
                    COALESCE(m.name, m.description) as name,
                    m.language as lang
                """

                result = session.run(query)
                self.entity_cache = [
                    (record["name"], record["label"], record["lang"])
                    for record in result
                    if record["name"]
                ]

                logger.info(f"Pre-cached {len(self.entity_cache)} entities")

        except Exception as e:
            logger.error(f"Error pre-caching entities: {e}")
            self.entity_cache = []

    @lru_cache(maxsize=1000)
    def _run_query_cached(self, query: str, params_hash: str) -> List[Dict]:
        """Cached query execution with LRU cache"""
        try:
            with self.driver.session() as session:
                result = session.run(query, json.loads(params_hash))
                return [record.data() for record in result]
        except Exception as e:
            logger.error(f"Database query error: {e}")
            return []

    def _run_query(self, query: str, params: Dict) -> List[Dict]:
        """Execute query with caching"""
        params_hash = json.dumps(params, sort_keys=True)
        return self._run_query_cached(query, params_hash)

    def _semantic_entity_linking(self, query: str, threshold: float = 0.7) -> List[Tuple[str, str, float]]:
        """Optimized semantic entity linking"""
        if not self.entity_cache:
            return []

        # Extract meaningful words
        words = re.findall(r'[\w\u0600-\u06FF]{3,}', query)
        if not words:
            return []

        # Batch process similarities
        entity_names = [entity[0] for entity in self.entity_cache]
        similarities = self.semantic_retriever.semantic_similarity_batch(words, entity_names)

        results = []
        for i, word in enumerate(words):
            for j, (entity_name, label, lang) in enumerate(self.entity_cache):
                if similarities[i, j] >= threshold:
                    results.append((entity_name, label, similarities[i, j]))

        # Deduplicate and sort
        unique_results = {}
        for entity, label, score in results:
            if entity not in unique_results or score > unique_results[entity][1]:
                unique_results[entity] = (label, score)

        return [(entity, label, score) for entity, (label, score) in unique_results.items()]

    def get_formulation_info(self, formulation_name: str) -> Optional[Dict]:
        """Optimized formulation query"""
        query = """
        MATCH (f:UnaniFormulation)
        WHERE toLower(f.name) CONTAINS toLower($name)
        OPTIONAL MATCH (f)-[:TREATS_DISEASE]->(d:Disease)
        OPTIONAL MATCH (f)-[:RELIEVES_SYMPTOM]->(s:Symptom)
        OPTIONAL MATCH (f)-[:HAS_BOTANICAL_NAME]->(b:BotanicalName)
        OPTIONAL MATCH (f)-[:USES_PLANT_PART]->(p:PlantPart)
        OPTIONAL MATCH (f)-[:HAS_TEMPERAMENT]->(t:Temperament)
        OPTIONAL MATCH (f)-[:CONTAINS_INGREDIENT]->(i:Ingredient)
        OPTIONAL MATCH (f)-[:HAS_DOSAGE]->(dos:Dosage)
        OPTIONAL MATCH (f)-[:HAS_TREATMENT]->(tr:Treatment)
        RETURN
            f.name as name,
            f.language as language,
            COLLECT(DISTINCT b.name) as botanic_names,
            COLLECT(DISTINCT t.name) as temperaments,
            COLLECT(DISTINCT p.name) as parts_used,
            COLLECT(DISTINCT i.name) as ingredients,
            COLLECT(DISTINCT dos.description) as dosages,
            COLLECT(DISTINCT tr.description) as treatments,
            COLLECT(DISTINCT d.name) as diseases,
            COLLECT(DISTINCT s.name) as symptoms
        LIMIT 1
        """
        results = self._run_query(query, {'name': formulation_name})
        return results[0] if results else None

    def get_graph_context(self, entities: List[str]) -> Dict:
        """Optimized context retrieval"""
        context = {
            'formulations': [],
            'treatments': [],
            'symptom_reliefs': [],
            'semantic_matches': []
        }

        # Process entities in parallel where possible
        with ThreadPoolExecutor(max_workers=min(4, len(entities))) as executor:
            # Submit all tasks
            future_to_entity = {}
            for entity in entities:
                future = executor.submit(self.get_formulation_info, entity)
                future_to_entity[future] = entity

            # Collect results
            for future in future_to_entity:
                try:
                    result = future.result(timeout=5.0)  # 5 second timeout
                    if result:
                        context['formulations'].append(result)
                except Exception as e:
                    logger.warning(f"Timeout processing entity: {e}")

        # Get semantic matches
        for entity in entities:
            semantic_matches = self._semantic_entity_linking(entity)
            if semantic_matches:
                context['semantic_matches'].extend(semantic_matches)

        return context

    def close(self):
        """Close the database connection"""
        self.driver.close()

class GraphRAGGenerator:
    """Optimized GraphRAG generator with efficient memory management"""

    def __init__(self, retriever: GraphRAGRetriever):
        self.retriever = retriever
        self._initialize_models()
        self.optimize_for_evaluation()

    def _initialize_models(self):
        """Initialize models with optimized settings"""
        with GLOBAL_MODELS_LOCK:
            # Tokenizer
            if 'tokenizer' not in GLOBAL_MODELS:
                self.tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
                self.tokenizer.pad_token = self.tokenizer.eos_token
                GLOBAL_MODELS['tokenizer'] = self.tokenizer
            else:
                self.tokenizer = GLOBAL_MODELS['tokenizer']

            # Model
            if 'model' not in GLOBAL_MODELS:
                bnb_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=torch.bfloat16,
                    bnb_4bit_use_double_quant=True,
                )

                self.model = AutoModelForCausalLM.from_pretrained(
                    BASE_MODEL,
                    device_map="auto",
                    quantization_config=bnb_config,
                    torch_dtype=torch.bfloat16,
                    low_cpu_mem_usage=True
                )

                # Load LoRA adapter if available
                if LORA_MODEL_PATH and os.path.exists(LORA_MODEL_PATH):
                    try:
                        self.model = PeftModel.from_pretrained(self.model, LORA_MODEL_PATH)
                        self.model = self.model.merge_and_unload()
                    except Exception as e:
                        logger.warning(f"Could not load LoRA adapter: {e}")

                GLOBAL_MODELS['model'] = self.model
            else:
                self.model = GLOBAL_MODELS['model']

  
        self.generator = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map="auto",
            max_new_tokens=196,  # Reduced for faster generation
            temperature=0.7,
            top_p=0.9,
            top_k=20,
            repetition_penalty=1.05,
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id,
            batch_size=1
        )

    def optimize_for_evaluation(self):
        """Optimize model for evaluation"""
        self.model.eval()
        torch.backends.cudnn.benchmark = True
        torch.cuda.empty_cache()

    def clear_memory(self):
        """Clear GPU memory"""
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        gc.collect()

    def _extract_entities(self, query: str) -> List[str]:
        """Fast entity extraction"""
        # Simple keyword extraction first
        stop_words = {'what', 'is', 'the', 'of', 'for', 'tell', 'me', 'about',
                     'کا', 'کی', 'کے', 'کہ', 'میں', 'سے', 'پر', 'کو', 'ہے', 'کیا'}

        words = re.findall(r'[\w\u0600-\u06FF]{3,}', query)
        entities = [word for word in words if word.lower() not in stop_words]

        return entities[:3]  # Limit to 3 most relevant entities

    def generate_response(self, query: str) -> Tuple[str, Dict, float]:
        """Optimized response generation"""
        start_time = time.time()

        lang = Language.detect(query)
        entities = self._extract_entities(query)


        try:
            with ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(self.retriever.get_graph_context, entities)
                graph_context = future.result(timeout=10.0)  # 10 second timeout
        except TimeoutError:
            logger.warning("Context retrieval timed out")
            graph_context = {}

        # Simplified prompt creation
        if lang == Language.URDU:
            prompt = self._create_urdu_prompt(query, graph_context)
        else:
            prompt = self._create_english_prompt(query, graph_context)

   
        try:
            with torch.inference_mode():
                response = self.generator(
                    prompt,
                    max_new_tokens=196,
                    temperature=0.7,
                    top_p=0.9
                )[0]['generated_text']

            response = response.replace(prompt, "").strip()

            if lang == Language.URDU:
                response = UrduTextFormatter.format(response)

        except Exception as e:
            logger.error(f"Generation error: {e}")
            response = "معذرت، میں اس سوال کا جواب نہیں دے سکتا۔" if lang == Language.URDU else "Sorry, I cannot answer this question."

        # Calculate confidence based on context richness
        confidence = 0.3
        if graph_context.get('formulations'):
            confidence += 0.3
        if graph_context.get('semantic_matches'):
            confidence += 0.2
        if len(entities) > 0:
            confidence += 0.2

        confidence = min(confidence, 0.95)

        logger.info(f"Response generated in {time.time() - start_time:.2f}s")

        return response, graph_context, confidence

    def _create_english_prompt(self, query: str, context: Dict) -> str:
        """Optimized English prompt"""
        prompt_parts = ["You are a Unani medicine expert. Answer based on:"]

        if context.get('formulations'):
            for formulation in context['formulations'][:2]:  # Limit to 2 formulations
                prompt_parts.append(f"\nFormulation: {formulation.get('name', 'N/A')}")
                for key in ['botanic_names', 'temperaments', 'parts_used', 'ingredients']:
                    if formulation.get(key):
                        prompt_parts.append(f"{key}: {', '.join(formulation[key][:3])}")  # Limit items

        prompt_parts.extend([
            f"\nQuestion: {query}",
            "\nProvide a concise answer:"
        ])

        return '\n'.join(prompt_parts)

    def _create_urdu_prompt(self, query: str, context: Dict) -> str:
        """Optimized Urdu prompt"""
        prompt_parts = ["آپ طب یونانی کے ماہر ہیں۔ درج ذیل معلومات کی روشنی میں جواب دیں:"]

        if context.get('formulations'):
            for formulation in context['formulations'][:2]:
                prompt_parts.append(f"\nفرمولیشن: {formulation.get('name', 'نام موجود نہیں')}")
                for key in ['botanic_names', 'temperaments', 'parts_used']:
                    if formulation.get(key):
                        prompt_parts.append(f"{key}: {', '.join(formulation[key][:3])}")

        prompt_parts.extend([
            f"\nسوال: {query}",
            "\nمختصر جواب دیں:"
        ])

        return '\n'.join(prompt_parts)

class UnaniGraphRAGSystem:
    """Optimized Unani Medicine Expert System"""

    def __init__(self):
        global GLOBAL_RETRIEVER
        if GLOBAL_RETRIEVER is None:
            GLOBAL_RETRIEVER = GraphRAGRetriever(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
        self.retriever = GLOBAL_RETRIEVER
        self.generator = GraphRAGGenerator(self.retriever)

    def query(self, question: str) -> Dict:
        """Optimized query processing"""
        response, context, confidence = self.generator.generate_response(question)

        return {
            'question': question,
            'answer': response,
            'confidence': f"{confidence:.0%}",
            'language': Language.detect(question),
            'processing_time': f"{time.time():.2f}s"
        }

    def close(self):
        """Close connections"""
        self.retriever.close()

def interactive_chat():
    """Optimized interactive chat"""
    print("Initializing Optimized Unani Medicine Expert System...")
    print("Type 'exit' to quit or type in English/Urdu\n")

    try:
        expert = UnaniGraphRAGSystem()

        while True:
            try:
                query = input("\nYou: ").strip()
                if not query:
                    continue

                # Handle exit command in both languages
                if query.lower() in ['exit', 'خروج', 'باہر', 'بند']:
                    print("System shutting down...")
                    break

                start_time = time.time()
                result = expert.query(query)
                processing_time = time.time() - start_time

                print(f"\nExpert ({result['language']}, {result['confidence']} confidence, {processing_time:.2f}s):")
                print("=" * 60)
                print(result['answer'])
                print("=" * 60)

            except KeyboardInterrupt:
                print("\nType 'exit' to quit")
            except Exception as e:
                print(f"Error: {e}")

    finally:
        expert.close()

if __name__ == "__main__":
   
    required_packages = {
        'arabic-reshaper': 'arabic_reshaper',
        'python-bidi': 'bidi',
        'sentence-transformers': 'sentence_transformers'
    }

    for package, import_name in required_packages.items():
        try:
            __import__(import_name)
        except ImportError:
            print(f"Installing {package}...")
            import subprocess
            subprocess.run(["pip", "install", package])

    interactive_chat()

Overwriting GRAPHRAG_MISTRAL.py


In [ ]:
!python GRAPHRAG_MISTRAL.py


2025-08-29 13:35:07.840525: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756474507.860932   16885 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756474507.867504   16885 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756474507.883328   16885 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756474507.883354   16885 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756474507.883358   16885 computation_placer.cc:177] computation placer alr

Mistral Base Model English Queries Evaluation

In [ ]:
#base model English
import json
import torch
import numpy as np
from tqdm import tqdm
import nltk
import time
from evaluate import load
import pandas as pd
from typing import Dict, List, Tuple
import gc


nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
from huggingface_hub import login


login(token="")

class UnaniEvaluator:
    def __init__(self, test_file: str, base_model_path: str = "mistralai/Mistral-7B-Instruct-v0.3",
                 fine_tuned_model_path: str = None):
        self.test_file = test_file
        self.base_model_path = base_model_path
        self.fine_tuned_model_path = fine_tuned_model_path
        self._load_test_data()

        # Initialize metrics
        self.bleu = load("bleu")
        self.rouge = load("rouge")
        self.meteor = load("meteor")
        self.bertscore = load("bertscore")

    def _load_test_data(self):
        """Load and preprocess test data"""
        with open(self.test_file, 'r', encoding='utf-8') as f:
            self.test_data = json.load(f)

        # Format test cases
        self.single_hop_cases = self.test_data["single_hop"]
        self.multi_hop_cases = self.test_data["multi_hop"]

        print(f"Loaded {len(self.single_hop_cases)} single-hop and {len(self.multi_hop_cases)} multi-hop test cases")

    def _format_prompt(self, instruction: str, system_prompt: str = None) -> str:
        """Format prompt in Mistral's instruction format"""
        if system_prompt is None:
            system_prompt = "You are an expert in Unani medicine. Provide accurate, detailed answers."

        return f"""<s>[INST] {system_prompt}

{instruction} [/INST]"""

    def load_base_model(self):
        """Load the base Mistral model"""
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

        self.base_tokenizer = AutoTokenizer.from_pretrained(
            self.base_model_path,
            padding_side="right",
            use_fast=False
        )
        self.base_tokenizer.pad_token = self.base_tokenizer.eos_token

        self.base_model = AutoModelForCausalLM.from_pretrained(
            self.base_model_path,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16
        )

        self.base_pipe = self._create_pipeline(self.base_model, self.base_tokenizer)
        print("Loaded base Mistral model")

    def load_fine_tuned_model(self):
        """Load the fine-tuned Mistral model"""
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import PeftModel

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

        self.ft_tokenizer = AutoTokenizer.from_pretrained(
            self.base_model_path,
            padding_side="right",
            use_fast=False
        )
        self.ft_tokenizer.pad_token = self.ft_tokenizer.eos_token

        base_model = AutoModelForCausalLM.from_pretrained(
            self.base_model_path,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16
        )

        self.ft_model = PeftModel.from_pretrained(base_model, self.fine_tuned_model_path)
        self.ft_model.eval()

        self.ft_pipe = self._create_pipeline(self.ft_model, self.ft_tokenizer)
        print("Loaded fine-tuned Mistral model")

    def _create_pipeline(self, model, tokenizer):
        """Create a text generation pipeline"""
        from transformers import pipeline

        return pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            temperature=0.3,
            max_new_tokens=256,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    def generate_with_model(self, prompt: str, pipe, model_type: str = "base") -> str:
        """Generate response using the specified model"""
        try:
            result = pipe(
                prompt,
                max_new_tokens=256,
                temperature=0.3,
                do_sample=True
            )

            
            generated_text = result[0]['generated_text']

           
            response = generated_text.replace(prompt, "").strip()

            return response
        except Exception as e:
            print(f"Generation failed for {model_type}: {str(e)}")
            return ""

    def generate_with_graphrag(self, query: str, graphrag_system) -> str:
        """Generate response using GraphRAG system"""
        try:
            result = graphrag_system.query(query)
            return result['answer']
        except Exception as e:
            print(f"GraphRAG generation failed: {str(e)}")
            return ""

    def calculate_metrics(self, predictions: List[str], references: List[str]) -> Dict:
        """Calculate all evaluation metrics"""
       
        preds = [pred if isinstance(pred, str) else ' '.join(pred) for pred in predictions]
        refs = [ref if isinstance(ref, str) else ' '.join(ref) for ref in references]

        # For BLEU, we need references as list of lists
        refs_for_bleu = [[ref] for ref in refs]

        
        metrics = {}

        # BLEU
        try:
            metrics["bleu"] = self.bleu.compute(predictions=preds, references=refs_for_bleu)["bleu"]
        except:
            metrics["bleu"] = 0.0

        # ROUGE
        rouge_results = self.rouge.compute(predictions=preds, references=refs)
        metrics["rouge1"] = rouge_results["rouge1"]
        metrics["rouge2"] = rouge_results["rouge2"]
        metrics["rougeL"] = rouge_results["rougeL"]

        # METEOR
        meteor_results = self.meteor.compute(predictions=preds, references=refs)
        metrics["meteor"] = meteor_results["meteor"]

        # BERTScore
        bertscore_results = self.bertscore.compute(predictions=preds, references=refs, lang="en")
        metrics["bertscore_precision"] = np.mean(bertscore_results["precision"])
        metrics["bertscore_recall"] = np.mean(bertscore_results["recall"])
        metrics["bertscore_f1"] = np.mean(bertscore_results["f1"])

        # Answer relevance (custom metric)
        rel_scores = [self.evaluate_answer_relevance(p, r) for p, r in zip(preds, refs)]
        metrics["answer_relevance"] = np.mean(rel_scores)

        return metrics

    def evaluate_answer_relevance(self, pred: str, ref: str) -> float:
        """Custom heuristic for answer relevance (0-1 scale)"""
        if not pred.strip():
            return 0.0

        pred_words = set(nltk.word_tokenize(pred.lower()))
        ref_words = set(nltk.word_tokenize(ref.lower()))

        # Jaccard similarity
        intersection = pred_words.intersection(ref_words)
        union = pred_words.union(ref_words)
        jaccard = len(intersection) / len(union) if union else 0

       
        key_terms = {"used", "treat", "recommended", "dosage", "temperature", "nature",
                    "botanical", "active", "ingredient", "symptom", "relieve", "condition"}
        term_score = sum(1 for term in key_terms if term in pred.lower()) / len(key_terms)

        return 0.7 * jaccard + 0.3 * term_score

    def evaluate_configuration(self, config_name: str, generate_func, test_cases: List[Dict],
                              case_type: str, progress_desc: str) -> Dict:
        """Evaluate a specific configuration"""
        predictions = []
        references = []
        processing_times = []

        for case in tqdm(test_cases, desc=progress_desc):
            prompt = self._format_prompt(case["instruction"])
            reference = case["output"]

            start_time = time.time()
            prediction = generate_func(case["instruction"])
            end_time = time.time()

            predictions.append(prediction)
            references.append(reference)
            processing_times.append(end_time - start_time)

        # Calculate metrics
        metrics = self.calculate_metrics(predictions, references)
        metrics["avg_processing_time"] = np.mean(processing_times)

        return {
            "predictions": predictions,
            "references": references,
            "metrics": metrics,
            "processing_times": processing_times
        }

    def evaluate_all_configurations(self):
        """Evaluate all four configurations"""
        results = {}

      
        print("\n" + "="*60)
        print("EVALUATING BASE MISTRAL MODEL")
        print("="*60)

        if hasattr(self, 'base_pipe'):
            base_func = lambda query: self.generate_with_model(
                self._format_prompt(query), self.base_pipe, "base"
            )

            # Single-hop
            sh_base = self.evaluate_configuration(
                "base_mistral", base_func, self.single_hop_cases,
                "single_hop", "Base Mistral - Single-hop"
            )

            # Multi-hop
            mh_base = self.evaluate_configuration(
                "base_mistral", base_func, self.multi_hop_cases,
                "multi_hop", "Base Mistral - Multi-hop"
            )

            results["base_mistral"] = {
                "single_hop": sh_base,
                "multi_hop": mh_base
            }

        # 2. Fine-tuned Mistral model
        print("\n" + "="*60)
        print("EVALUATING FINE-TUNED MISTRAL MODEL")
        print("="*60)

        if hasattr(self, 'ft_pipe'):
            ft_func = lambda query: self.generate_with_model(
                self._format_prompt(query), self.ft_pipe, "fine_tuned"
            )

       
            sh_ft = self.evaluate_configuration(
                "ft_mistral", ft_func, self.single_hop_cases,
                "single_hop", "Fine-tuned Mistral - Single-hop"
            )

         
            mh_ft = self.evaluate_configuration(
                "ft_mistral", ft_func, self.multi_hop_cases,
                "multi_hop", "Fine-tuned Mistral - Multi-hop"
            )

            results["ft_mistral"] = {
                "single_hop": sh_ft,
                "multi_hop": mh_ft
            }

       
        print("\n" + "="*60)
        print("EVALUATING GRAPHRAG WITH BASE MISTRAL")
        print("="*60)

        try:
            from GRAPHRAG_MISTRAL import UnaniGraphRAGSystem
            graphrag_system = UnaniGraphRAGSystem()

            graphrag_func = lambda query: self.generate_with_graphrag(query, graphrag_system)

            # Single-hop
            sh_graphrag = self.evaluate_configuration(
                "graphrag_base", graphrag_func, self.single_hop_cases,
                "single_hop", "GraphRAG Base - Single-hop"
            )

            # Multi-hop
            mh_graphrag = self.evaluate_configuration(
                "graphrag_base", graphrag_func, self.multi_hop_cases,
                "multi_hop", "GraphRAG Base - Multi-hop"
            )

            results["graphrag_base"] = {
                "single_hop": sh_graphrag,
                "multi_hop": mh_graphrag
            }

            graphrag_system.close()
        except Exception as e:
            print(f"Failed to initialize GraphRAG: {e}")

        # 4. GraphRAG with fine-tuned Mistral
        print("\n" + "="*60)
        print("EVALUATING GRAPHRAG WITH FINE-TUNED MISTRAL")
        print("="*60)

        try:
            from GRAPHRAG_MISTRAL import UnaniGraphRAGSystem

            graphrag_system = UnaniGraphRAGSystem()

            graphrag_func = lambda query: self.generate_with_graphrag(query, graphrag_system)

          
            sh_graphrag_ft = self.evaluate_configuration(
                "graphrag_ft", graphrag_func, self.single_hop_cases,
                "single_hop", "GraphRAG FT - Single-hop"
            )

        
            mh_graphrag_ft = self.evaluate_configuration(
                "graphrag_ft", graphrag_func, self.multi_hop_cases,
                "multi_hop", "GraphRAG FT - Multi-hop"
            )

            results["graphrag_ft"] = {
                "single_hop": sh_graphrag_ft,
                "multi_hop": mh_graphrag_ft
            }

            graphrag_system.close()
        except Exception as e:
            print(f"Failed to initialize GraphRAG with fine-tuned model: {e}")

        return results

    def print_results(self, results: Dict):
        """Print formatted evaluation results"""
        print("\n" + "="*80)
        print("COMPREHENSIVE UNANI MEDICINE MODEL EVALUATION RESULTS")
        print("="*80)


        summary_data = []

        for config_name, config_results in results.items():
            for case_type in ["single_hop", "multi_hop"]:
                if case_type in config_results:
                    metrics = config_results[case_type]["metrics"]

                    summary_data.append({
                        "Configuration": config_name,
                        "Case Type": case_type,
                        "BLEU": f"{metrics.get('bleu', 0):.4f}",
                        "ROUGE-1": f"{metrics.get('rouge1', 0):.4f}",
                        "ROUGE-2": f"{metrics.get('rouge2', 0):.4f}",
                        "ROUGE-L": f"{metrics.get('rougeL', 0):.4f}",
                        "METEOR": f"{metrics.get('meteor', 0):.4f}",
                        "BERTScore F1": f"{metrics.get('bertscore_f1', 0):.4f}",
                        "BERTScore Precision": f"{metrics.get('bertscore_precision', 0):.4f}",
                        "BERTScore Recall": f"{metrics.get('bertscore_recall', 0):.4f}",
                        "Answer Relevance": f"{metrics.get('answer_relevance', 0):.4f}",
                        "Avg Time (s)": f"{metrics.get('avg_processing_time', 0):.2f}"
                    })


        df = pd.DataFrame(summary_data)
        print("\nSummary Table:")
        print(df.to_string(index=False))

        # Print detailed results for each configuration
        for config_name, config_results in results.items():
            print(f"\n\n{'='*60}")
            print(f"DETAILED RESULTS FOR {config_name.upper()}")
            print(f"{'='*60}")

            for case_type in ["single_hop", "multi_hop"]:
                if case_type in config_results:
                    print(f"\n{case_type.upper()} RESULTS:")
                    print("-" * 40)

                    metrics = config_results[case_type]["metrics"]

                    print(f"BLEU: {metrics.get('bleu', 0):.4f}")
                    print(f"ROUGE-1: {metrics.get('rouge1', 0):.4f}")
                    print(f"ROUGE-2: {metrics.get('rouge2', 0):.4f}")
                    print(f"ROUGE-L: {metrics.get('rougeL', 0):.4f}")
                    print(f"METEOR: {metrics.get('meteor', 0):.4f}")
                    print(f"BERTScore F1: {metrics.get('bertscore_f1', 0):.4f}")
                    print(f"BERTScore Precision: {metrics.get('bertscore_precision', 0):.4f}")
                    print(f"BERTScore Recall: {metrics.get('bertscore_recall', 0):.4f}")
                    print(f"Answer Relevance: {metrics.get('answer_relevance', 0):.4f}")
                    print(f"Average Processing Time: {metrics.get('avg_processing_time', 0):.2f}s")

              
                    if case_type == "single_hop":
                        sample_idx = 0
                    else:
                        sample_idx = 0

                    sample_case = self.single_hop_cases[sample_idx] if case_type == "single_hop" else self.multi_hop_cases[sample_idx]
                    sample_pred = config_results[case_type]["predictions"][sample_idx]

                    print(f"\nSample Input: {sample_case['instruction']}")
                    print(f"Reference Output: {sample_case['output']}")
                    print(f"Model Output: {sample_pred}")

    def save_results(self, results: Dict, filename: str = "unani_comprehensive_evaluation_results.json"):
      
      
        def convert_numpy_types(obj):
            if isinstance(obj, np.integer):
                return int(obj)
            elif isinstance(obj, np.floating):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, dict):
                return {k: convert_numpy_types(v) for k, v in obj.items()}
            elif isinstance(obj, list):
                return [convert_numpy_types(item) for item in obj]
            else:
                return obj

        results = convert_numpy_types(results)

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        print(f"\nResults saved to {filename}")

def main():

    TEST_FILE = "final.json"
    BASE_MODEL_PATH = "mistralai/Mistral-7B-Instruct-v0.3"
    FINE_TUNED_MODEL_PATH = ""  
    # Initialize evaluator
    evaluator = UnaniEvaluator(TEST_FILE, BASE_MODEL_PATH, FINE_TUNED_MODEL_PATH)

  
    print("Loading models...")
    evaluator.load_base_model()

    if FINE_TUNED_MODEL_PATH:
        evaluator.load_fine_tuned_model()

    # Run evaluation
    print("Starting evaluation...")
    results = evaluator.evaluate_all_configurations()

    # Print and save results
    evaluator.print_results(results)
    evaluator.save_results(results)

    # Clean up
    gc.collect()
    torch.cuda.empty_cache()

if __name__ == "__main__":
    main()

Loaded 52 single-hop and 47 multi-hop test cases


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Loading models...


tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

GraphRAG FT - Single-hop:  98%|█████████▊| 51/52 [00:16<00:00,  2.58it/s]

GraphRAG generation failed: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 2146 has 14.74 GiB memory in use. Of the allocated memory 14.57 GiB is allocated by PyTorch, and 33.18 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


GraphRAG FT - Single-hop: 100%|██████████| 52/52 [00:17<00:00,  3.05it/s]

GraphRAG generation failed: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 2146 has 14.74 GiB memory in use. Of the allocated memory 14.57 GiB is allocated by PyTorch, and 33.18 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Failed to initialize GraphRAG with fine-tuned model: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 2146 has 14.74 GiB memory in use. Of the allocated memory 14.57 GiB is allocated by PyTorch, and 33.18 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

COMPREHENSIVE UNANI MEDICINE MODEL EVALUATION RESULTS

Summary Table:
Configuration  Case Type   BLEU ROUGE-1 ROUGE-2 ROUGE-L METEOR BERTScore F1 BERTScore Precision BERTScore Recall Answer Relevance Avg Time (s)
 base_mistral single_hop 0.0112  0.0870  0.0371  0.0768 0.1854       0.8380              0.8009           0.8789           0.1388        13.44
 base_mistral  multi_hop 0.0058  0.1035  0.0143  0.0763 0.1924       0.8187  

Mistral Base Model Urdu Queries Evaluation

In [ ]:
#base model Urdu
import json
import torch
import numpy as np
from tqdm import tqdm
import nltk
import time
from evaluate import load
import pandas as pd
from typing import Dict, List, Tuple
import gc

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
from huggingface_hub import login

login(token="")

class UnaniEvaluator:
    def __init__(self, test_file: str, base_model_path: str = "mistralai/Mistral-7B-Instruct-v0.3",
                 fine_tuned_model_path: str = None):
        self.test_file = test_file
        self.base_model_path = base_model_path
        self.fine_tuned_model_path = fine_tuned_model_path
        self._load_test_data()

        # Initialize metrics
        self.bleu = load("bleu")
        self.rouge = load("rouge")
        self.meteor = load("meteor")
        self.bertscore = load("bertscore")

    def _load_test_data(self):
        """Load and preprocess test data"""
        with open(self.test_file, 'r', encoding='utf-8') as f:
            self.test_data = json.load(f)

        # Format test cases
        self.single_hop_cases = self.test_data["single_hop"]
        self.multi_hop_cases = self.test_data["multi_hop"]

        print(f"Loaded {len(self.single_hop_cases)} single-hop and {len(self.multi_hop_cases)} multi-hop test cases")

    def _format_prompt(self, instruction: str, system_prompt: str = None) -> str:
        """Format prompt in Mistral's instruction format"""
        if system_prompt is None:
            system_prompt = "You are an expert in Unani medicine. Provide accurate, detailed answers."

        return f"""<s>[INST] {system_prompt}

{instruction} [/INST]"""

    def load_base_model(self):
        """Load the base Mistral model"""
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

        self.base_tokenizer = AutoTokenizer.from_pretrained(
            self.base_model_path,
            padding_side="right",
            use_fast=False
        )
        self.base_tokenizer.pad_token = self.base_tokenizer.eos_token

        self.base_model = AutoModelForCausalLM.from_pretrained(
            self.base_model_path,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16
        )

        self.base_pipe = self._create_pipeline(self.base_model, self.base_tokenizer)
        print("Loaded base Mistral model")

    def load_fine_tuned_model(self):
        """Load the fine-tuned Mistral model"""
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import PeftModel

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

        self.ft_tokenizer = AutoTokenizer.from_pretrained(
            self.base_model_path,
            padding_side="right",
            use_fast=False
        )
        self.ft_tokenizer.pad_token = self.ft_tokenizer.eos_token

        base_model = AutoModelForCausalLM.from_pretrained(
            self.base_model_path,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16
        )

        self.ft_model = PeftModel.from_pretrained(base_model, self.fine_tuned_model_path)
        self.ft_model.eval()

        self.ft_pipe = self._create_pipeline(self.ft_model, self.ft_tokenizer)
        print("Loaded fine-tuned Mistral model")

    def _create_pipeline(self, model, tokenizer):
        """Create a text generation pipeline"""
        from transformers import pipeline

        return pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            temperature=0.3,
            max_new_tokens=256,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    def generate_with_model(self, prompt: str, pipe, model_type: str = "base") -> str:
        """Generate response using the specified model"""
        try:
            result = pipe(
                prompt,
                max_new_tokens=256,
                temperature=0.3,
                do_sample=True
            )

   
            generated_text = result[0]['generated_text']

         
            response = generated_text.replace(prompt, "").strip()

            return response
        except Exception as e:
            print(f"Generation failed for {model_type}: {str(e)}")
            return ""

    def generate_with_graphrag(self, query: str, graphrag_system) -> str:
        """Generate response using GraphRAG system"""
        try:
            result = graphrag_system.query(query)
            return result['answer']
        except Exception as e:
            print(f"GraphRAG generation failed: {str(e)}")
            return ""

    def calculate_metrics(self, predictions: List[str], references: List[str]) -> Dict:
        """Calculate all evaluation metrics"""
      
        preds = [pred if isinstance(pred, str) else ' '.join(pred) for pred in predictions]
        refs = [ref if isinstance(ref, str) else ' '.join(ref) for ref in references]

    
        refs_for_bleu = [[ref] for ref in refs]

     
        metrics = {}

     
        try:
            metrics["bleu"] = self.bleu.compute(predictions=preds, references=refs_for_bleu)["bleu"]
        except:
            metrics["bleu"] = 0.0

  
        rouge_results = self.rouge.compute(predictions=preds, references=refs)
        metrics["rouge1"] = rouge_results["rouge1"]
        metrics["rouge2"] = rouge_results["rouge2"]
        metrics["rougeL"] = rouge_results["rougeL"]

        
        meteor_results = self.meteor.compute(predictions=preds, references=refs)
        metrics["meteor"] = meteor_results["meteor"]

  
        bertscore_results = self.bertscore.compute(predictions=preds, references=refs, lang="en")
        metrics["bertscore_precision"] = np.mean(bertscore_results["precision"])
        metrics["bertscore_recall"] = np.mean(bertscore_results["recall"])
        metrics["bertscore_f1"] = np.mean(bertscore_results["f1"])


        rel_scores = [self.evaluate_answer_relevance(p, r) for p, r in zip(preds, refs)]
        metrics["answer_relevance"] = np.mean(rel_scores)

        return metrics

    def evaluate_answer_relevance(self, pred: str, ref: str) -> float:
        """Custom heuristic for answer relevance (0-1 scale)"""
        if not pred.strip():
            return 0.0

        pred_words = set(nltk.word_tokenize(pred.lower()))
        ref_words = set(nltk.word_tokenize(ref.lower()))

       
        intersection = pred_words.intersection(ref_words)
        union = pred_words.union(ref_words)
        jaccard = len(intersection) / len(union) if union else 0

     
        key_terms = {"used", "treat", "recommended", "dosage", "temperature", "nature",
                    "botanical", "active", "ingredient", "symptom", "relieve", "condition"}
        term_score = sum(1 for term in key_terms if term in pred.lower()) / len(key_terms)

        return 0.7 * jaccard + 0.3 * term_score

    def evaluate_configuration(self, config_name: str, generate_func, test_cases: List[Dict],
                              case_type: str, progress_desc: str) -> Dict:
        """Evaluate a specific configuration"""
        predictions = []
        references = []
        processing_times = []

        for case in tqdm(test_cases, desc=progress_desc):
            prompt = self._format_prompt(case["instruction"])
            reference = case["output"]

            start_time = time.time()
            prediction = generate_func(case["instruction"])
            end_time = time.time()

            predictions.append(prediction)
            references.append(reference)
            processing_times.append(end_time - start_time)

  
        metrics = self.calculate_metrics(predictions, references)
        metrics["avg_processing_time"] = np.mean(processing_times)

        return {
            "predictions": predictions,
            "references": references,
            "metrics": metrics,
            "processing_times": processing_times
        }

    def evaluate_all_configurations(self):
        """Evaluate all four configurations"""
        results = {}

    
        print("\n" + "="*60)
        print("EVALUATING BASE MISTRAL MODEL")
        print("="*60)

        if hasattr(self, 'base_pipe'):
            base_func = lambda query: self.generate_with_model(
                self._format_prompt(query), self.base_pipe, "base"
            )

     
            sh_base = self.evaluate_configuration(
                "base_mistral", base_func, self.single_hop_cases,
                "single_hop", "Base Mistral - Single-hop"
            )

    
            mh_base = self.evaluate_configuration(
                "base_mistral", base_func, self.multi_hop_cases,
                "multi_hop", "Base Mistral - Multi-hop"
            )

            results["base_mistral"] = {
                "single_hop": sh_base,
                "multi_hop": mh_base
            }


        print("\n" + "="*60)
        print("EVALUATING FINE-TUNED MISTRAL MODEL")
        print("="*60)

        if hasattr(self, 'ft_pipe'):
            ft_func = lambda query: self.generate_with_model(
                self._format_prompt(query), self.ft_pipe, "fine_tuned"
            )


            sh_ft = self.evaluate_configuration(
                "ft_mistral", ft_func, self.single_hop_cases,
                "single_hop", "Fine-tuned Mistral - Single-hop"
            )


            mh_ft = self.evaluate_configuration(
                "ft_mistral", ft_func, self.multi_hop_cases,
                "multi_hop", "Fine-tuned Mistral - Multi-hop"
            )

            results["ft_mistral"] = {
                "single_hop": sh_ft,
                "multi_hop": mh_ft
            }

        print("\n" + "="*60)
        print("EVALUATING GRAPHRAG WITH BASE MISTRAL")
        print("="*60)

        try:
            from GRAPHRAG_MISTRAL import UnaniGraphRAGSystem
            graphrag_system = UnaniGraphRAGSystem()

            graphrag_func = lambda query: self.generate_with_graphrag(query, graphrag_system)

  
            sh_graphrag = self.evaluate_configuration(
                "graphrag_base", graphrag_func, self.single_hop_cases,
                "single_hop", "GraphRAG Base - Single-hop"
            )

       
            mh_graphrag = self.evaluate_configuration(
                "graphrag_base", graphrag_func, self.multi_hop_cases,
                "multi_hop", "GraphRAG Base - Multi-hop"
            )

            results["graphrag_base"] = {
                "single_hop": sh_graphrag,
                "multi_hop": mh_graphrag
            }

            graphrag_system.close()
        except Exception as e:
            print(f"Failed to initialize GraphRAG: {e}")

        # 4. GraphRAG with fine-tuned Mistral
        print("\n" + "="*60)
        print("EVALUATING GRAPHRAG WITH FINE-TUNED MISTRAL")
        print("="*60)

        try:
            from GRAPHRAG_MISTRAL import UnaniGraphRAGSystem

            graphrag_system = UnaniGraphRAGSystem()

            graphrag_func = lambda query: self.generate_with_graphrag(query, graphrag_system)

     
            sh_graphrag_ft = self.evaluate_configuration(
                "graphrag_ft", graphrag_func, self.single_hop_cases,
                "single_hop", "GraphRAG FT - Single-hop"
            )

            mh_graphrag_ft = self.evaluate_configuration(
                "graphrag_ft", graphrag_func, self.multi_hop_cases,
                "multi_hop", "GraphRAG FT - Multi-hop"
            )

            results["graphrag_ft"] = {
                "single_hop": sh_graphrag_ft,
                "multi_hop": mh_graphrag_ft
            }

            graphrag_system.close()
        except Exception as e:
            print(f"Failed to initialize GraphRAG with fine-tuned model: {e}")

        return results

    def print_results(self, results: Dict):
        """Print formatted evaluation results"""
        print("\n" + "="*80)
        print("COMPREHENSIVE UNANI MEDICINE MODEL EVALUATION RESULTS")
        print("="*80)

  
        summary_data = []

        for config_name, config_results in results.items():
            for case_type in ["single_hop", "multi_hop"]:
                if case_type in config_results:
                    metrics = config_results[case_type]["metrics"]

                    summary_data.append({
                        "Configuration": config_name,
                        "Case Type": case_type,
                        "BLEU": f"{metrics.get('bleu', 0):.4f}",
                        "ROUGE-1": f"{metrics.get('rouge1', 0):.4f}",
                        "ROUGE-2": f"{metrics.get('rouge2', 0):.4f}",
                        "ROUGE-L": f"{metrics.get('rougeL', 0):.4f}",
                        "METEOR": f"{metrics.get('meteor', 0):.4f}",
                        "BERTScore F1": f"{metrics.get('bertscore_f1', 0):.4f}",
                        "BERTScore Precision": f"{metrics.get('bertscore_precision', 0):.4f}",
                        "BERTScore Recall": f"{metrics.get('bertscore_recall', 0):.4f}",
                        "Answer Relevance": f"{metrics.get('answer_relevance', 0):.4f}",
                        "Avg Time (s)": f"{metrics.get('avg_processing_time', 0):.2f}"
                    })


        df = pd.DataFrame(summary_data)
        print("\nSummary Table:")
        print(df.to_string(index=False))


        for config_name, config_results in results.items():
            print(f"\n\n{'='*60}")
            print(f"DETAILED RESULTS FOR {config_name.upper()}")
            print(f"{'='*60}")

            for case_type in ["single_hop", "multi_hop"]:
                if case_type in config_results:
                    print(f"\n{case_type.upper()} RESULTS:")
                    print("-" * 40)

                    metrics = config_results[case_type]["metrics"]

                    print(f"BLEU: {metrics.get('bleu', 0):.4f}")
                    print(f"ROUGE-1: {metrics.get('rouge1', 0):.4f}")
                    print(f"ROUGE-2: {metrics.get('rouge2', 0):.4f}")
                    print(f"ROUGE-L: {metrics.get('rougeL', 0):.4f}")
                    print(f"METEOR: {metrics.get('meteor', 0):.4f}")
                    print(f"BERTScore F1: {metrics.get('bertscore_f1', 0):.4f}")
                    print(f"BERTScore Precision: {metrics.get('bertscore_precision', 0):.4f}")
                    print(f"BERTScore Recall: {metrics.get('bertscore_recall', 0):.4f}")
                    print(f"Answer Relevance: {metrics.get('answer_relevance', 0):.4f}")
                    print(f"Average Processing Time: {metrics.get('avg_processing_time', 0):.2f}s")


                    if case_type == "single_hop":
                        sample_idx = 0
                    else:
                        sample_idx = 0

                    sample_case = self.single_hop_cases[sample_idx] if case_type == "single_hop" else self.multi_hop_cases[sample_idx]
                    sample_pred = config_results[case_type]["predictions"][sample_idx]

                    print(f"\nSample Input: {sample_case['instruction']}")
                    print(f"Reference Output: {sample_case['output']}")
                    print(f"Model Output: {sample_pred}")

    def save_results(self, results: Dict, filename: str = "unani_comprehensive_evaluation_results.json"):


        def convert_numpy_types(obj):
            if isinstance(obj, np.integer):
                return int(obj)
            elif isinstance(obj, np.floating):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, dict):
                return {k: convert_numpy_types(v) for k, v in obj.items()}
            elif isinstance(obj, list):
                return [convert_numpy_types(item) for item in obj]
            else:
                return obj

        results = convert_numpy_types(results)

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        print(f"\nResults saved to {filename}")

def main():
    # Configuration
    TEST_FILE = "final_urdu.json"
    BASE_MODEL_PATH = "mistralai/Mistral-7B-Instruct-v0.3"
    FINE_TUNED_MODEL_PATH = ""  
    # Initialize evaluator
    evaluator = UnaniEvaluator(TEST_FILE, BASE_MODEL_PATH, FINE_TUNED_MODEL_PATH)

    # Load models
    print("Loading models...")
    evaluator.load_base_model()

    if FINE_TUNED_MODEL_PATH:
        evaluator.load_fine_tuned_model()


    print("Starting evaluation...")
    results = evaluator.evaluate_all_configurations()


    evaluator.print_results(results)
    evaluator.save_results(results)

    # Clean up
    gc.collect()
    torch.cuda.empty_cache()

if __name__ == "__main__":
    main()

Loaded 56 single-hop and 33 multi-hop test cases


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Loading models...


tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

GraphRAG FT - Single-hop:  98%|█████████▊| 55/56 [00:16<00:00,  3.58it/s]

GraphRAG generation failed: CUDA out of memory. Tried to allocate 12.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 4.12 MiB is free. Process 4544 has 14.73 GiB memory in use. Of the allocated memory 14.56 GiB is allocated by PyTorch, and 37.72 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


GraphRAG FT - Single-hop: 100%|██████████| 56/56 [00:16<00:00,  3.34it/s]

GraphRAG generation failed: CUDA out of memory. Tried to allocate 12.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 4.12 MiB is free. Process 4544 has 14.73 GiB memory in use. Of the allocated memory 14.56 GiB is allocated by PyTorch, and 37.82 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Failed to initialize GraphRAG with fine-tuned model: CUDA out of memory. Tried to allocate 40.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 4.12 MiB is free. Process 4544 has 14.73 GiB memory in use. Of the allocated memory 14.56 GiB is allocated by PyTorch, and 37.58 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

COMPREHENSIVE UNANI MEDICINE MODEL EVALUATION RESULTS

Summary Table:
Configuration  Case Type   BLEU ROUGE-1 ROUGE-2 ROUGE-L METEOR BERTScore F1 BERTScore Precision BERTScore Recall Answer Relevance Avg Time (s)
 base_mistral single_hop 0.0010  0.0014  0.0000  0.0014 0.0504       0.7657              0.7800           0.7531           0.0655        14.60
 base_mistral  multi_hop 0.0062  0.0067  0.0015  0.0066 0.0891       0.8069 

Mistral GRAPHRAG Model English Queries Evaluation

In [ ]:
#GRAPHRAG model English
import json
import torch
import numpy as np
from tqdm import tqdm
import nltk
import time
from evaluate import load
import pandas as pd
from typing import Dict, List, Tuple
import gc


nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

class GraphRAGEvaluator:
    def __init__(self, test_file: str):
        self.test_file = test_file
        self._load_test_data()


        self.bleu = load("bleu")
        self.rouge = load("rouge")
        self.meteor = load("meteor")
        self.bertscore = load("bertscore")

    def _load_test_data(self):
        """Load and preprocess test data"""
        with open(self.test_file, 'r', encoding='utf-8') as f:
            self.test_data = json.load(f)

        self.single_hop_cases = self.test_data["single_hop"]
        self.multi_hop_cases = self.test_data["multi_hop"]

        print(f"Loaded {len(self.single_hop_cases)} single-hop and {len(self.multi_hop_cases)} multi-hop test cases")

    def generate_with_graphrag(self, query: str, graphrag_system) -> str:
        """Generate response using GraphRAG system"""
        try:
            result = graphrag_system.query(query)
            return result['answer']
        except Exception as e:
            print(f"GraphRAG generation failed: {str(e)}")
            return ""

    def calculate_metrics(self, predictions: List[str], references: List[str]) -> Dict:
        """Calculate all evaluation metrics"""
        
        preds = [pred if isinstance(pred, str) else ' '.join(pred) for pred in predictions]
        refs = [ref if isinstance(ref, str) else ' '.join(ref) for ref in references]

  
        refs_for_bleu = [[ref] for ref in refs]


        metrics = {}


        try:
            metrics["bleu"] = self.bleu.compute(predictions=preds, references=refs_for_bleu)["bleu"]
        except:
            metrics["bleu"] = 0.0

      
        rouge_results = self.rouge.compute(predictions=preds, references=refs)
        metrics["rouge1"] = rouge_results["rouge1"]
        metrics["rouge2"] = rouge_results["rouge2"]
        metrics["rougeL"] = rouge_results["rougeL"]

        
        meteor_results = self.meteor.compute(predictions=preds, references=refs)
        metrics["meteor"] = meteor_results["meteor"]

        
        bertscore_results = self.bertscore.compute(predictions=preds, references=refs, lang="en")
        metrics["bertscore_precision"] = np.mean(bertscore_results["precision"])
        metrics["bertscore_recall"] = np.mean(bertscore_results["recall"])
        metrics["bertscore_f1"] = np.mean(bertscore_results["f1"])

   
        rel_scores = [self.evaluate_answer_relevance(p, r) for p, r in zip(preds, refs)]
        metrics["answer_relevance"] = np.mean(rel_scores)

        return metrics

    def evaluate_answer_relevance(self, pred: str, ref: str) -> float:
        
        if not pred.strip():
            return 0.0

        pred_words = set(nltk.word_tokenize(pred.lower()))
        ref_words = set(nltk.word_tokenize(ref.lower()))

   
        intersection = pred_words.intersection(ref_words)
        union = pred_words.union(ref_words)
        jaccard = len(intersection) / len(union) if union else 0

      
        key_terms = {"used", "treat", "recommended", "dosage", "temperature", "nature",
                    "botanical", "active", "ingredient", "symptom", "relieve", "condition"}
        term_score = sum(1 for term in key_terms if term in pred.lower()) / len(key_terms)

        return 0.7 * jaccard + 0.3 * term_score

    def evaluate_graphrag(self, test_cases: List[Dict], case_type: str, progress_desc: str) -> Dict:
        """Evaluate GraphRAG system"""
        predictions = []
        references = []
        processing_times = []


        from GRAPHRAG_MISTRAL import UnaniGraphRAGSystem
        graphrag_system = UnaniGraphRAGSystem()

        try:
            for case in tqdm(test_cases, desc=progress_desc):
                reference = case["output"]

                start_time = time.time()
                prediction = self.generate_with_graphrag(case["instruction"], graphrag_system)
                end_time = time.time()

                predictions.append(prediction)
                references.append(reference)
                processing_times.append(end_time - start_time)

   
            metrics = self.calculate_metrics(predictions, references)
            metrics["avg_processing_time"] = np.mean(processing_times)

            return {
                "predictions": predictions,
                "references": references,
                "metrics": metrics,
                "processing_times": processing_times
            }
        finally:
            graphrag_system.close()

    def evaluate_graphrag_configurations(self):
        """Evaluate GraphRAG system on both single-hop and multi-hop cases"""
        results = {}

        print("\n" + "="*60)
        print("EVALUATING GRAPHRAG WITH FINE-TUNED MISTRAL")
        print("="*60)


        sh_graphrag = self.evaluate_graphrag(
            self.single_hop_cases,
            "single_hop",
            "GraphRAG FT - Single-hop"
        )

  
        mh_graphrag = self.evaluate_graphrag(
            self.multi_hop_cases,
            "multi_hop",
            "GraphRAG FT - Multi-hop"
        )

        results["graphrag_ft"] = {
            "single_hop": sh_graphrag,
            "multi_hop": mh_graphrag
        }

        return results

    def print_results(self, results: Dict):
        """Print formatted evaluation results"""
        print("\n" + "="*80)
        print("GRAPH RAG WITH FINE-TUNED MISTRAL EVALUATION RESULTS")
        print("="*80)


        summary_data = []

        for config_name, config_results in results.items():
            for case_type in ["single_hop", "multi_hop"]:
                if case_type in config_results:
                    metrics = config_results[case_type]["metrics"]

                    summary_data.append({
                        "Configuration": config_name,
                        "Case Type": case_type,
                        "BLEU": f"{metrics.get('bleu', 0):.4f}",
                        "ROUGE-1": f"{metrics.get('rouge1', 0):.4f}",
                        "ROUGE-2": f"{metrics.get('rouge2', 0):.4f}",
                        "ROUGE-L": f"{metrics.get('rougeL', 0):.4f}",
                        "METEOR": f"{metrics.get('meteor', 0):.4f}",
                        "BERTScore F1": f"{metrics.get('bertscore_f1', 0):.4f}",
                        "BERTScore Precision": f"{metrics.get('bertscore_precision', 0):.4f}",
                        "BERTScore Recall": f"{metrics.get('bertscore_recall', 0):.4f}",
                        "Answer Relevance": f"{metrics.get('answer_relevance', 0):.4f}",
                        "Avg Time (s)": f"{metrics.get('avg_processing_time', 0):.2f}"
                    })

     
        df = pd.DataFrame(summary_data)
        print("\nSummary Table:")
        print(df.to_string(index=False))


        for config_name, config_results in results.items():
            print(f"\n\n{'='*60}")
            print(f"DETAILED RESULTS FOR {config_name.upper()}")
            print(f"{'='*60}")

            for case_type in ["single_hop", "multi_hop"]:
                if case_type in config_results:
                    print(f"\n{case_type.upper()} RESULTS:")
                    print("-" * 40)

                    metrics = config_results[case_type]["metrics"]

                    print(f"BLEU: {metrics.get('bleu', 0):.4f}")
                    print(f"ROUGE-1: {metrics.get('rouge1', 0):.4f}")
                    print(f"ROUGE-2: {metrics.get('rouge2', 0):.4f}")
                    print(f"ROUGE-L: {metrics.get('rougeL', 0):.4f}")
                    print(f"METEOR: {metrics.get('meteor', 0):.4f}")
                    print(f"BERTScore F1: {metrics.get('bertscore_f1', 0):.4f}")
                    print(f"BERTScore Precision: {metrics.get('bertscore_precision', 0):.4f}")
                    print(f"BERTScore Recall: {metrics.get('bertscore_recall', 0):.4f}")
                    print(f"Answer Relevance: {metrics.get('answer_relevance', 0):.4f}")
                    print(f"Average Processing Time: {metrics.get('avg_processing_time', 0):.2f}s")

                    
                    sample_indices = [0, 1, 2] 

                    for i, sample_idx in enumerate(sample_indices):
                        if case_type == "single_hop" and sample_idx < len(self.single_hop_cases):
                            sample_case = self.single_hop_cases[sample_idx]
                        elif case_type == "multi_hop" and sample_idx < len(self.multi_hop_cases):
                            sample_case = self.multi_hop_cases[sample_idx]
                        else:
                            continue

                        sample_pred = config_results[case_type]["predictions"][sample_idx]

                        print(f"\nSample {i+1}:")
                        print(f"Input: {sample_case['instruction']}")
                        print(f"Reference Output: {sample_case['output']}")
                        print(f"Model Output: {sample_pred}")
                        print("-" * 40)

    def save_results(self, results: Dict, filename: str = "graphrag_evaluation_results.json"):
        """Save evaluation results to a JSON file"""
       
        def convert_numpy_types(obj):
            if isinstance(obj, np.integer):
                return int(obj)
            elif isinstance(obj, np.floating):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, dict):
                return {k: convert_numpy_types(v) for k, v in obj.items()}
            elif isinstance(obj, list):
                return [convert_numpy_types(item) for item in obj]
            else:
                return obj

        results = convert_numpy_types(results)

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        print(f"\nResults saved to {filename}")

def main():

    TEST_FILE = "final.json"

    evaluator = GraphRAGEvaluator(TEST_FILE)


    print("Starting GraphRAG evaluation...")
    results = evaluator.evaluate_graphrag_configurations()


    evaluator.print_results(results)
    evaluator.save_results(results)


    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if __name__ == "__main__":
    main()

Loaded 52 single-hop and 47 multi-hop test cases


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
Device set to use cuda:0


Starting GraphRAG evaluation...

EVALUATING GRAPHRAG WITH FINE-TUNED MISTRAL


GraphRAG FT - Single-hop:   2%|▏         | 1/52 [00:53<45:46, 53.84s/it]ERROR:neo4j.pool:Unable to retrieve routing information
ERROR:GRAPHRAG_MISTRAL:Database query error: Unable to retrieve routing information
ERROR:neo4j.pool:Unable to retrieve routing information
ERROR:GRAPHRAG_MISTRAL:Database query error: Unable to retrieve routing information
ERROR:neo4j.pool:Unable to retrieve routing information
ERROR:GRAPHRAG_MISTRAL:Database query error: Unable to retrieve routing information
GraphRAG FT - Single-hop:  19%|█▉        | 10/52 [08:09<32:36, 46.58s/it]WARNING:GRAPHRAG_MISTRAL:Context retrieval timed out
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
GraphRAG FT - Single-hop: 100%|██████████| 52/52 [42:41<00:00, 49.26s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this mod


GRAPH RAG WITH FINE-TUNED MISTRAL EVALUATION RESULTS

Summary Table:
Configuration  Case Type   BLEU ROUGE-1 ROUGE-2 ROUGE-L METEOR BERTScore F1 BERTScore Precision BERTScore Recall Answer Relevance Avg Time (s)
  graphrag_ft single_hop 0.0202  0.1712  0.0607  0.1587 0.2893       0.8649              0.8272           0.9067           0.1454        49.26
  graphrag_ft  multi_hop 0.0052  0.1320  0.0216  0.0997 0.2088       0.8295              0.8119           0.8481           0.1209        55.33


DETAILED RESULTS FOR GRAPHRAG_FT

SINGLE_HOP RESULTS:
----------------------------------------
BLEU: 0.0202
ROUGE-1: 0.1712
ROUGE-2: 0.0607
ROUGE-L: 0.1587
METEOR: 0.2893
BERTScore F1: 0.8649
BERTScore Precision: 0.8272
BERTScore Recall: 0.9067
Answer Relevance: 0.1454
Average Processing Time: 49.26s

Sample 1:
Input: What is the botanical name of Gaozaban-e-Hindi?
Reference Output: The botanical name of Gaozaban-e-Hindi is Onosma echioides.
Model Output: The botanical name of Gaozaban-e-Hindi 